# Pronóstico de demanda de bicicletas

Notebook integrador: EDA, estadística, validación temporal, modelos clásicos, red neuronal y consultas para el tablero. Ejecuta desde la raíz del repositorio.

El modelo predice el volumen por hora. Se excluyen `casual` y `registered` porque suman el objetivo `count`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))
import pandas as pd
from common import load_observations
df = load_observations()
df.head()

## 1. Calidad y resumen de los datos
Revisa dimensiones, tipos, valores faltantes, duplicados y huecos en la secuencia horaria antes de modelar. Los huecos de tiempo no se convierten en demanda cero.


In [ ]:
print('Filas y columnas:', df.shape)
display(df.isna().sum().to_frame('faltantes'))
print('Duplicados por fecha/hora:', df.duplicated('datetime').sum())
expected_hours = pd.date_range(df.datetime.min(), df.datetime.max(), freq='h')
missing_hours = expected_hours.difference(pd.DatetimeIndex(df.datetime))
print('Horas sin registro en la secuencia esperada:', len(missing_hours))
display(df.describe(include='all').T)


## 2. Exploración y estadística
Observa patrones por hora y temporada. El contraste entre días laborales y no laborales es asociativo; no implica causalidad.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
display(df.groupby('hour')['count'].agg(['mean','median','std']).round(1))
sns.lineplot(data=df, x='hour', y='count', estimator='mean', errorbar=None, marker='o')
plt.title('Demanda promedio por hora'); plt.xlabel('Hora'); plt.ylabel('Alquileres por hora'); plt.show()
display(df.groupby('season')['count'].agg(['mean','median','std']).round(1))

## 3. Modelos clásicos con validación cruzada temporal
Se comparan regresión lineal, árbol y Random Forest con cinco divisiones que respetan el orden del tiempo. El último 20% cronológico queda como prueba final.

In [ ]:
from train_models import main
metrics = main()
display(pd.read_csv(ROOT / 'outputs/metrics/comparacion_modelos.csv'))
metrics['best_classical_model']

## 4. Red neuronal y control de sobreajuste
El script entrena una red densa en Keras y detiene el entrenamiento cuando la pérdida de validación deja de mejorar. Se compara con el mejor modelo clásico usando el mismo tramo de prueba.

In [ ]:
metrics['neural_network']
from IPython.display import Image, display
img = ROOT / 'outputs/figures/keras_early_stopping.png'
if img.exists(): display(Image(filename=str(img)))

## 5. SQL, modelo estrella y Power BI
La siguiente celda crea las tablas DimDate, DimHour, DimConditions y FactDemand para SQLite y archivos CSV de carga en Power BI.

In [ ]:
from build_sqlite import main as build_sqlite
build_sqlite()

## 6. Asistente generativo
El agente puede usar el predictor como herramienta. Sin API key, devuelve una respuesta local de demostración y lo identifica.

In [ ]:
from demand_agent import generate_summary
example = {'datetime':'2012-07-10 08:00','season':3,'holiday':0,'workingday':1,'weather':1,'temp':28.0,'atemp':31.0,'humidity':62.0,'windspeed':11.0}
generate_summary(example)